# middleware

In [12]:
import os
from langchain_openai import ChatOpenAI
#获取api_key
api_key = os.getenv('ARK_API_KEY')
model = ChatOpenAI(
    openai_api_base="https://ark.cn-beijing.volces.com/api/v3",
    openai_api_key=api_key,	# app_key
    model_name="doubao-seed-1-6-flash-250828",	# 您想使用的特定模型的名称或标识符。
    max_tokens=5000, #限制响应中的令牌总数，有效控制输出长度。
    temperature= 0.7,  #控制模型输出的随机性。值越高，响应越具创造性；值越低，响应越确定性。
    timeout=30, #模型的响应时间s
)
model1 = ChatOpenAI(
    openai_api_base="https://ark.cn-beijing.volces.com/api/v3",
    openai_api_key=api_key,	# app_key
    model_name="doubao-seed-1-6-251015",	# 您想使用的特定模型的名称或标识符。
    max_tokens=5000, #限制响应中的令牌总数，有效控制输出长度。
    temperature= 0.7,  #控制模型输出的随机性。值越高，响应越具创造性；值越低，响应越确定性。
    timeout=30, #模型的响应时间s
)

In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model,
    tools=[],
    middleware=[
        SummarizationMiddleware(
            model,
            trigger=("tokens", 4000),
            message_to_keep=20,
            summary_prompt="Custom prompt for summarization..."
        ),
    ]
)

In [6]:
# Human in the loop
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import ModelCallLimitMiddleware, ToolCallLimitMiddleware, ModelFallbackMiddleware
from langchain.agents.middleware import PIIMiddleware

# 限制所有工具调用
global_limiter = ToolCallLimitMiddleware(thread_limit=20, run_limit=10)

# 限制特定工具
search_limiter = ToolCallLimitMiddleware(
    tool_name="search",
    thread_limit=5,
    run_limit=3,
)

agent = create_agent(
    model,
    tools=[],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                #要求对方发送邮件进行批准，编辑或者拒绝
                "send_email_tool":{
                    "allowed_decisions":["approved","edit","reject"],
                },
                #自动批准读取邮件
                "read_email_tool": False,
            }
        ),
        ModelCallLimitMiddleware(
            thread_limit=10,  # 每个线程（跨多次运行）最多 10 次调用
            run_limit=5,  # 每次运行（单次调用）最多 5 次调用
            exit_behavior="end",  # 或者 "error" 以引发异常
        ),
        global_limiter,search_limiter,
        ModelFallbackMiddleware(
            model,  # 错误时首先尝试
            model,  # 然后尝试这个
        ),
        #涂改用户电子邮件
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        # 掩盖信用卡（显示后4位）
        PIIMiddleware("credit_card", strategy="mast", apply_to_input=True),
        # 带有正则表达式的自定义PII类型
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block"
        ),
    ],
)

In [10]:
from langchain.agents import create_agent
from langchain.agents.middleware import TodoListMiddleware
from langchain.messages import HumanMessage
from langchain.agents.middleware import LLMToolSelectorMiddleware, ToolRetryMiddleware


agent = create_agent(
    model=model,
    tools=[],
    middleware=[TodoListMiddleware(),
               LLMToolSelectorMiddleware(
                   model = model1,
                   max_tools=3,  # 限制为 3 个最相关的工具
            always_include=["search"],  # 始终包含某些工具
               ),
                ToolRetryMiddleware(
                max_retries=3,  # 最多重试 3 次
                backoff_factor=2.0,  # 指数回退乘数
                initial_delay=1.0,  # 从 1 秒延迟开始
                max_delay=60.0,  # 将延迟上限设置为 60 秒
                jitter=True,  # 添加随机抖动以避免“惊群”问题
                ),
               ],
)

result = agent.invoke({"messages": [HumanMessage("Help me refactor my codebase")]})
print(result["todos"])  # 带有状态跟踪的待办事项数组

[{'content': 'Gather information about the codebase from the user (e.g., programming language, project structure, known issues, refactoring goals, current pain points)', 'status': 'in_progress'}, {'content': 'Assess the current codebase to identify refactoring needs and pain points (e.g., duplicated code, naming issues, performance bottlenecks)', 'status': 'pending'}, {'content': 'Create a prioritized refactoring plan with specific, actionable tasks (e.g., fix critical technical debt first, then improve readability)', 'status': 'pending'}, {'content': 'Start with small, non-breaking refactoring tasks (e.g., variable renaming, extracting small functions)', 'status': 'pending'}, {'content': 'Implement each small refactoring task and test code functionality immediately after completion', 'status': 'pending'}, {'content': 'Refactor larger components (modules, classes) based on the prioritized plan', 'status': 'pending'}, {'content': 'Run comprehensive test suites to ensure all functionalit

In [17]:
from langchain.agents import create_agent
from langchain.agents.middleware import LLMToolEmulator


agent = create_agent(
    model,
    tools=[get_weather, search_database, send_email],
    middleware=[
        # 默认模拟所有工具
        LLMToolEmulator(),

        # 或模拟特定工具
        # LLMToolEmulator(tools=["get_weather", "search_database"]),

        # 或使用自定义模型进行模拟
        # LLMToolEmulator(model="anthropic:claude-3-5-sonnet-latest"),
    ],
)
agent.invoke({"messages":[{"role":"user","content":"what is weather in SF?"}]})

NameError: name 'get_weather' is not defined

In [21]:
# 自定义中间件
from langchain.agents.middleware import before_model, after_model, wrap_model_call
from langchain.agents.middleware import AgentState, ModelRequest, ModelResponse, dynamic_prompt
from langgraph.runtime import Runtime
from langchain.messages import AIMessage
from typing import Any, Callable



#Node-style 模型调用前的日志记录
@before_model
def log_before_model(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    print(f"About to call model with {len(state['messages'])} messages")
    return None

#Node-style 模型调用后的验证
@after_model(can_jump_to=["end"])
def validate_output(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    last_message = state["messages"][-1]
    if "BLOCKED" in last_message:
        return {
            "messages": [AIMessage("I cannot respond to that request.")],
            "jump_to": "end"
        }
    return None

@wrap_model_call
def retry_model(request: ModelRequest, response: ModelResponse) -> ModelResponse:
    for attempt in range(3):
        try:
            return handler(request)
        except Exception as e:
            if attempt == 2:
                raise
            print(f"Retry {attempt +1}/3 after error: {e}")

@dynamic_prompt
def personalized_prompt(request: ModelRequest) -> str:
    user_id = request.runtime.context.get("user_id", "guest")
    return f"You are a helpful assistant for user {user_id}. Be concise and friendly."
    
agent = create_agent(
    model,
    middleware=[log_before_model,validate_output,retry_model,personalized_prompt],
)
agent.invoke({"messages": [HumanMessage("你能作什么？")]})

About to call model with 1 messages
Retry 1/3 after error: name 'handler' is not defined
Retry 2/3 after error: name 'handler' is not defined


NameError: name 'handler' is not defined